# Taller Interactivo: Movimiento de Proyectiles y Aplicaciones en Ingeniería
### Escuela Colombiana de Ingeniería Julio Garavito — Física Mecánica
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
**Versión para Google Colab.** Cada punto es una celda **autónoma**: puedes ejecutar
cualquiera sin depender de las demás. Los sliders y el quiz vienen incluidos.

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Si un slider no aparece, vuelve a
> ejecutar esa celda.

Gravedad usada:  g = 9,81 m/s²  (sin resistencia del aire)

---
## Punto 1 — Lanzamiento de un dron de carga (Ing. Mecánica/Aeronáutica)
Un proyectil (paquete lanzado por catapulta o dron) sale con rapidez  v₀ = 20,0 m/s
formando un ángulo  θ = 45°  con la horizontal, desde el suelo.

**Ecuaciones (tiro parabólico):**
- Componentes:  vₓ = v₀·cos(θ),   v_y = v₀·sen(θ)
- Tiempo de vuelo:  t = 2·v₀·sen(θ) / g
- Alcance:  R = v₀²·sen(2θ) / g
- Altura máxima:  H = v₀²·sen²(θ) / (2·g)

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


g = 9.81

def tiro(v0=20.0, ang=45.0):
    th = np.radians(ang)
    vx = v0*np.cos(th); vy = v0*np.sin(th)
    t_fl = 2*vy/g
    R = v0**2*np.sin(2*th)/g
    H = vy**2/(2*g)
    t = np.linspace(0, t_fl, 200)
    x = vx*t; y = vy*t - 0.5*g*t**2
    plt.figure(figsize=(9,5))
    plt.plot(x, y, lw=2, color='navy')
    plt.scatter([R],[0], color='red', s=60, zorder=5, label=f'alcance={R:.1f} m')
    plt.scatter([R/2],[H], color='green', s=60, zorder=5, label=f'H_max={H:.1f} m')
    plt.xlabel('distancia horizontal (m)'); plt.ylabel('altura (m)')
    plt.title(f'Trayectoria: v₀={v0:.0f} m/s, θ={ang:.0f}°')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5)
    plt.tight_layout(); plt.show()
    print(f"Componentes: vₓ={vx:.2f} m/s, v_y={vy:.2f} m/s")
    print(f"Tiempo de vuelo = {t_fl:.3f} s")
    print(f"Alcance R = {R:.3f} m")
    print(f"Altura máxima H = {H:.3f} m")

interact(tiro,
    v0=FloatSlider(value=20.0, min=5, max=50, step=1, description='v₀ (m/s)'),
    ang=FloatSlider(value=45.0, min=10, max=80, step=5, description='θ (grados)'));

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("a) Tiempo de vuelo (v₀=20 m/s, θ=45°):", 2.883, 0.04, "s",
    "t = 2·v₀·sen θ / g = 2·20·0,707/9,81 ≈ 2,88 s.")
quiz_numerico("a) Alcance R (v₀=20 m/s, θ=45°):", 40.775, 0.04, "m",
    "R = v₀²·sen(2θ)/g = 400·1/9,81 ≈ 40,8 m.")
quiz_opcion_multiple("b) ¿En qué punto la velocidad vertical v_y es cero?",
    ["Al inicio", "En la altura máxima", "Al caer al suelo", "Nunca"],
    1, "En el punto más alto v_y=0; solo queda la componente horizontal vₓ.")


---
## Punto 2 — Entrega desde un dron en vuelo (Ing. Civil/Logística)
Un paquete se suelta con velocidad **horizontal**  v₀ = 15,0 m/s  desde una altura
h = 45,0 m  (dron o edificio). Cae por gravedad mientras avanza.

**Ecuaciones (movimiento horizontal + caída libre):**
- Tiempo de caída:  t = √(2·h / g)
- Alcance horizontal:  x = v₀·t
- Velocidad de impacto:  v = √(v₀² + (g·t)²)

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


g = 9.81

def caida(v0=15.0, h=45.0):
    t_fall = np.sqrt(2*h/g)
    x_reach = v0*t_fall
    vy_imp = g*t_fall
    v_imp = np.hypot(v0, vy_imp)
    t = np.linspace(0, t_fall, 200)
    x = v0*t; y = h - 0.5*g*t**2
    plt.figure(figsize=(9,5))
    plt.plot(x, y, lw=2, color='darkgreen')
    plt.scatter([x_reach],[0], color='red', s=60, zorder=5, label=f'impacto x={x_reach:.1f} m')
    plt.xlabel('distancia horizontal (m)'); plt.ylabel('altura (m)')
    plt.title(f'Lanzamiento horizontal: v₀={v0:.0f} m/s desde h={h:.0f} m')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5)
    plt.tight_layout(); plt.show()
    print(f"Tiempo de caída = {t_fall:.3f} s")
    print(f"Alcance horizontal = {x_reach:.3f} m")
    print(f"Velocidad de impacto = {v_imp:.3f} m/s")

interact(caida,
    v0=FloatSlider(value=15.0, min=5, max=40, step=1, description='v₀ (m/s)'),
    h=FloatSlider(value=45.0, min=10, max=100, step=5, description='h (m)'));

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("a) Tiempo de caída (h=45 m):", 3.029, 0.04, "s",
    "t = √(2h/g) = √(90/9,81) ≈ 3,03 s. No depende de v₀.")
quiz_numerico("a) Alcance horizontal (v₀=15 m/s, h=45 m):", 45.434, 0.04, "m",
    "x = v₀·t = 15·3,03 ≈ 45,4 m.")
quiz_opcion_multiple("b) Si sueltas el paquete con MAYOR velocidad horizontal, el tiempo de caída:",
    ["Aumenta", "Disminuye", "No cambia: la caída vertical es independiente de v₀", "Se duplica"],
    2, "El tiempo de caída solo depende de la altura h; la velocidad horizontal no afecta la caída vertical.")


---
## Punto 3 — Ángulo óptimo de alcance (Ing. Industrial/Deportiva)
Con rapidez de salida fija  v₀ = 20,0 m/s, se busca el ángulo que da el **mayor alcance**
(diseño de lanzadores, riego, trayectorias óptimas).

**Alcance:**  R = v₀²·sen(2θ) / g

El alcance es máximo cuando sen(2θ) = 1, es decir  2θ = 90°  →  θ = 45°.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


g = 9.81

def alcance_vs_angulo(v0=20.0, ang_marcado=45.0):
    angs = np.linspace(0, 90, 300)
    R = v0**2*np.sin(2*np.radians(angs))/g
    Rm = v0**2*np.sin(2*np.radians(ang_marcado))/g
    plt.figure(figsize=(9,5))
    plt.plot(angs, R, lw=2, color='purple')
    plt.axvline(45, color='green', ls='--', label='óptimo 45°')
    plt.scatter([ang_marcado],[Rm], color='red', s=60, zorder=5, label=f'θ={ang_marcado:.0f}° → {Rm:.1f} m')
    plt.xlabel('ángulo θ (grados)'); plt.ylabel('alcance R (m)')
    plt.title(f'Alcance vs ángulo (v₀={v0:.0f} m/s)')
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
    print(f"Alcance en θ={ang_marcado:.0f}°: R = {Rm:.3f} m")
    print(f"Alcance máximo (θ=45°): R = {v0**2/g:.3f} m")
    # Comparar angulos complementarios
    comp = 90 - ang_marcado
    Rc = v0**2*np.sin(2*np.radians(comp))/g
    print(f"Ángulo complementario θ={comp:.0f}° da el MISMO alcance: {Rc:.3f} m")

interact(alcance_vs_angulo,
    v0=FloatSlider(value=20.0, min=5, max=50, step=1, description='v₀ (m/s)'),
    ang_marcado=FloatSlider(value=45.0, min=5, max=85, step=5, description='θ (grados)'));

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("a) Alcance máximo (v₀=20 m/s, θ=45°):", 40.775, 0.04, "m",
    "R_max = v₀²/g = 400/9,81 ≈ 40,8 m.")
quiz_opcion_multiple("a) ¿Qué ángulo da el alcance máximo (sin aire)?",
    ["30°", "45°", "60°", "90°"], 1,
    "R = v₀²·sen(2θ)/g es máximo cuando sen(2θ)=1, o sea θ=45°.")
quiz_opcion_multiple("b) ¿Qué ángulos dan el MISMO alcance?",
    ["Ninguno", "Ángulos complementarios (ej. 30° y 60°)", "Solo 45°", "Ángulos iguales"],
    1, "Ángulos complementarios (θ y 90°−θ) dan el mismo alcance: 30° y 60°, 20° y 70°, etc.")


---
## ✅ Fin del taller de Proyectiles
Cada celda es independiente. Si un slider no se muestra, re-ejecuta esa celda.
*Física Mecánica — 2026-2.*